# Andrew Ng Digital Twin: cloned-voice TTS on a Kaggle GPURuns the Chatterbox voice-cloning model on Kaggle's free T4 and exposes it toyour local machine over a tunnel.**Why this is needed:** Chatterbox is a neural TTS model. On CPU it takes 10 to30 seconds per sentence, so a ten-sentence answer means minutes before thefirst word. It needs a GPU.```your laptop                    Kaggle (this notebook, T4)-----------                    --------------------------frontend + backend  --tunnel-->  Chatterbox TTSNeon Postgres```## Before running1. **Settings > Accelerator > GPU T4 x2**2. **Settings > Internet > On** (required, the tunnel cannot open without it)3. Upload `backend/data/andrew_ng_ref.wav` as a **private** Kaggle Dataset   named `andrew-ng-voice`, then attach it with *Add Input*.Run every cell top to bottom. The last cell prints the URL to paste into yourlocal `.env`, then keeps running to hold the session open.

In [ ]:
# 1. Dependencies. Takes a few minutes on first run.!pip install -q chatterbox-tts fastapi uvicorn soundfile nest_asyncio# cloudflared gives a public https URL with no account or token.!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64!chmod +x cloudflaredprint("dependencies ready")

In [ ]:
# 2. Configuration and checksimport os# Path if you attached the dataset as `andrew-ng-voice`.REFERENCE_AUDIO_PATH = "/kaggle/input/andrew-ng-voice/andrew_ng_ref.wav"# Alternative: a direct download URL instead of a Kaggle dataset.REFERENCE_AUDIO_URL = ""PORT = 5002if REFERENCE_AUDIO_URL and not os.path.exists(REFERENCE_AUDIO_PATH):    REFERENCE_AUDIO_PATH = "/kaggle/working/reference.wav"    os.system("wget -q -O " + REFERENCE_AUDIO_PATH + " " + REFERENCE_AUDIO_URL)if not os.path.exists(REFERENCE_AUDIO_PATH):    print("REFERENCE AUDIO NOT FOUND at", REFERENCE_AUDIO_PATH)    print("")    print("Available inputs:")    for root, dirs, files in os.walk("/kaggle/input"):        for f in files:            print("   ", os.path.join(root, f))    raise SystemExit("Attach the voice dataset, or set REFERENCE_AUDIO_URL above.")size_mb = os.path.getsize(REFERENCE_AUDIO_PATH) / 1_000_000print("reference audio:", REFERENCE_AUDIO_PATH, "(%.1f MB)" % size_mb)import torchDEVICE = "cuda" if torch.cuda.is_available() else "cpu"if DEVICE == "cpu":    print("")    print("WARNING: no GPU detected. Settings > Accelerator > GPU T4 x2.")    print("On CPU this will be too slow to be usable.")else:    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 3. Watermarker## Embeds an inaudible provenance marker so generated audio can later be# identified as synthetic. Kept ENABLED deliberately: for cloned speech of a# real, identifiable person, stripping it is indefensible. See docs/POSTURE.md.WATERMARKING = Truetry:    import perth    print("watermarking enabled")except Exception as exc:    WATERMARKING = False    import sys    from unittest.mock import MagicMock    class _Passthrough:        def __init__(self, *a, **k):            pass        def __call__(self, wav, *a, **k):            return wav        def encode(self, wav, *a, **k):            return wav        def apply_watermark(self, wav, *a, **k):            return wav    _m = MagicMock()    _m.PerthImplicitWatermarker = _Passthrough    _m.DummyWatermarker = _Passthrough    sys.modules["perth"] = _m    print("WARNING: perth unavailable (%s)" % exc)    print("Generated audio will NOT carry a synthetic-speech marker.")

In [ ]:
# 4. Load the model. First run downloads weights, a few minutes.from chatterbox.tts import ChatterboxTTSprint("loading Chatterbox...")MODEL = ChatterboxTTS.from_pretrained(device=DEVICE)print("ready on", DEVICE, "| sample rate", MODEL.sr)

In [ ]:
# 5. Quick check before wiring anything up. You should hear the cloned voice.import timefrom IPython.display import Audio, displayt0 = time.time()wav = MODEL.generate(    "So the key idea here is that gradient descent just follows the slope downhill.",    audio_prompt_path=REFERENCE_AUDIO_PATH,)print("synthesised in %.1fs" % (time.time() - t0))display(Audio(wav.squeeze(0).cpu().numpy(), rate=MODEL.sr))

In [ ]:
# 6. HTTP server.## Same request and response contract as run_chatterbox_server.py, so the# backend cannot tell local and remote apart.import ioimport timeimport soundfile as sffrom fastapi import FastAPI, HTTPExceptionfrom fastapi.responses import Responsefrom pydantic import BaseModelapp = FastAPI(title="Chatterbox TTS (Kaggle GPU)")class SpeechRequest(BaseModel):    model: str = "tts-1"    input: str    voice: str = "andrew_ng_ref"    speed: float = 1.0@app.get("/health")def health():    return {"status": "ok", "device": DEVICE,            "watermarking": WATERMARKING, "voice": "andrew_ng_ref"}@app.post("/v1/audio/speech")def speech(req: SpeechRequest):    text = (req.input or "").strip()    if not text:        raise HTTPException(status_code=400, detail="Input text cannot be empty")    started = time.time()    try:        kwargs = {"audio_prompt_path": REFERENCE_AUDIO_PATH}        # Speed at synthesis time, never by resampling finished audio, which        # would shift pitch and formants and undo the point of cloning.        if abs(req.speed - 1.0) > 0.01:            try:                kwargs["cfg_weight"] = max(0.2, min(1.0, 0.5 / req.speed))                wav = MODEL.generate(text, **kwargs)            except TypeError:                wav = MODEL.generate(text, audio_prompt_path=REFERENCE_AUDIO_PATH)        else:            wav = MODEL.generate(text, **kwargs)        buf = io.BytesIO()        sf.write(buf, wav.squeeze(0).cpu().numpy(), MODEL.sr,                 format="WAV", subtype="PCM_16")        buf.seek(0)        print("  %4d chars in %.1fs" % (len(text), time.time() - started))        return Response(content=buf.read(), media_type="audio/wav")    except Exception as exc:        print("  generation failed:", exc)        raise HTTPException(status_code=500, detail=str(exc))print("server defined")

In [ ]:
# 7. Start the server and open the tunnel.## This cell does not finish on purpose. It holds the session open. Leave the# tab open; Kaggle stops idle sessions.import subprocessimport threadingimport timeimport nest_asyncioimport uvicornnest_asyncio.apply()threading.Thread(    target=lambda: uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning"),    daemon=True,).start()time.sleep(4)tunnel = subprocess.Popen(    ["./cloudflared", "tunnel", "--url", "http://localhost:%d" % PORT, "--no-autoupdate"],    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,)public_url = Nonefor line in tunnel.stdout:    if "trycloudflare.com" in line:        for tok in line.split():            if tok.startswith("https://") and "trycloudflare" in tok:                public_url = tok.strip()                break    if public_url:        breakif not public_url:    raise SystemExit("Tunnel failed to start. Check Settings > Internet is On.")print("")print("=" * 70)print("TTS SERVER READY")print("=" * 70)print("")print("device:      ", DEVICE)print("watermarking:", "enabled" if WATERMARKING else "DISABLED")print("")print("1. Put this in your local .env:")print("")print("       CHATTERBOX_URL=" + public_url + "/v1/audio/speech")print("")print("2. Restart the backend.")print("")print("3. Check it from your laptop:")print("")print("       curl " + public_url + "/health")print("")print("Keep this tab open. The URL changes every time this cell is re-run.")print("=" * 70)try:    while True:        time.sleep(60)except KeyboardInterrupt:    tunnel.terminate()    print("stopped")

## Once it is runningIn your local `.env`:```bashCHATTERBOX_URL=https://<printed>.trycloudflare.com/v1/audio/speech```Restart the backend, then verify the whole chain from your laptop:```bashpython scripts/smoke_test.py```The services section should show `cloned voice ok, device=cuda`.## Session limitsKaggle sessions expire after about 9 hours, the URL changes on every restart,and the tab must stay open. This is a development and demo setup, notproduction.If re-pasting the URL becomes tiresome, use ngrok with a static domain (thefree tier includes one), which gives a permanent address:```python!pip install -q pyngrokfrom pyngrok import ngrokngrok.set_auth_token("YOUR_TOKEN")tunnel = ngrok.connect(PORT, domain="your-static-domain.ngrok-free.app")print(tunnel.public_url)```## If the voice service is downNothing breaks. The frontend checks `/api/v1/chat/tts/status` and falls back tothe browser's own speech synthesis, giving a generic voice instead of theclone. Voice mode keeps working either way.